# Visualisierung Ergebnisse Forschungsfrage 1

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns

datei = Path("../data/results_forschungsfrage1.csv")
data = pd.read_csv(datei, sep=";", encoding="utf-8-sig")
data["zeitpunkt"] = pd.to_datetime(data["zeitpunkt"])

# Filtern nach den aktuellsten Ergebnissen und n10000
data = (
    data[
        (data["status"] == "ok") &
        (data["variante"] == "n10000")
    ]
    .sort_values("zeitpunkt")
    .drop_duplicates(
        subset=["slug", "variante", "modell"],
        keep="last"
    )
    .reset_index(drop=True)
)

data

In [ ]:
data["performance_rank"] = (
    data.groupby("slug")
    .apply(
        lambda g: g["wert_mittel"].rank(
            ascending=g["richtung"].iloc[0] != 1,
            method="min"
        ),
        include_groups=False
    )
    .reset_index(level=0, drop=True)
)

data["time_rank"] = (
    data.groupby("slug")["sekunden_gesamt"]
    .rank(method="min")
)

data

In [ ]:
def plot_metric(metric="auc", figsize=(13, 8)):

    # Daten filtern
    df = data[
    (data["metrik_code"] == metric)
].copy()

    if df.empty:
        print(f"Keine Daten für Metrik '{metric}' gefunden.")
        return

    df["wert_mittel"] = pd.to_numeric(df["wert_mittel"])
    df["wert_std"] = pd.to_numeric(df["wert_std"])

    # Datensätze und Modelle
    datasets = sorted(df["slug"].unique())
    models = sorted(df["modell"].unique())

    # Farben
    colors = plt.cm.tab10(np.linspace(0, 1, len(models)))
    color_map = dict(zip(models, colors))

    # Modellpositionen innerhalb eines Datensatzes
    offsets = np.linspace(-0.25, 0.25, len(models))

    fig, ax = plt.subplots(figsize=figsize)

    # Datenpunkte
    for i, dataset in enumerate(datasets):

        for j, modell in enumerate(models):

            row = df[
                (df["slug"] == dataset) &
                (df["modell"] == modell)
            ]

            if row.empty:
                continue

            row = row.iloc[0]

            ax.errorbar(
                row["wert_mittel"],
                i + offsets[j],
                xerr=row["wert_std"],
                fmt="o",
                markersize=8,
                capsize=4,
                color=color_map[modell],
                ecolor=color_map[modell],
                label=modell if i == 0 else None,
                zorder=3
            )

    # Datensatz-Trennlinien
    for i in range(len(datasets) - 1):
        ax.axhline(
            i + 0.5,
            color="gray",
            linewidth=1,
            alpha=0.7,
            zorder=1
        )

    # Abwechselnde Hintergrundfarben
    for i in range(len(datasets)):
        if i % 2 == 0:
            ax.axhspan(
                i - 0.5,
                i + 0.5,
                color="gray",
                alpha=0.07,
                zorder=0
            )

    # Achsen
    ax.set_yticks(range(len(datasets)))
    ax.set_yticklabels(datasets)
    ax.invert_yaxis()

    ax.set_xlabel(metric.upper(), fontsize=12)
    ax.set_ylabel("Datensatz", fontsize=12)

    ax.set_title(
        f"Modellvergleich – {metric.upper()}",
        fontsize=15,
        fontweight="bold"
    )

    # X-Achse automatisch skalieren
    xmin = (df["wert_mittel"] - df["wert_std"]).min()
    xmax = (df["wert_mittel"] + df["wert_std"]).max()

    padding = max((xmax - xmin) * 0.08, 0.005)
    ax.set_xlim(xmin - padding, xmax + padding)

    ax.grid(axis="x", alpha=0.3, zorder=0)

    # Legende
    ax.legend(title="Modell", loc="best")

    plt.tight_layout()
    plt.show()


In [ ]:
plot_metric("auc")
plot_metric("accuracy")
plot_metric("auc_multilabel")
plot_metric("rmse")
plot_metric("map3")
plot_metric("balanced_accuracy")
plot_metric("r2")
plot_metric("mcc")

In [ ]:

models = ["rf", "xgb", "tabfm", "tabpfn"]
ranks = [1, 2, 3, 4]

# Farbpalette: Dunkelrot bis Hellrot
colors = [
    "#B05A5A",  # Platz 1
    "#C47D7D",  # Platz 2
    "#D9A5A5",  # Platz 3
    "#EBD0D0"   # Platz 4
]

# Häufigkeiten berechnen
performance_counts = (
    data.groupby(["modell", "performance_rank"])
    .size()
    .unstack(fill_value=0)
    .reindex(index=models, columns=ranks, fill_value=0)
)

time_counts = (
    data.groupby(["modell", "time_rank"])
    .size()
    .unstack(fill_value=0)
    .reindex(index=models, columns=ranks, fill_value=0)
)


def plot_ranking(counts, title):
    ax = counts.plot(
        kind="bar",
        figsize=(10, 5),
        width=0.8,
        color=colors
    )

    ax.set_title(title, fontsize=14, pad=15)
    ax.set_xlabel("")
    ax.set_ylabel("Anzahl Datensätze")
    ax.set_xticklabels(models, rotation=0)

    ax.legend(
        [f"Platz {i}" for i in ranks],
        title="Ranking",
        frameon=False
    )

    ax.grid(axis="y", alpha=0.25)
    ax.set_axisbelow(True)

    # Rahmen entfernen
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()
    plt.show()


plot_ranking(
    performance_counts,
    "Performance-Ranking"
)

plot_ranking(
    time_counts,
    "Laufzeit-Ranking"
)